In [ ]:
NUM_ANTS = 50                   # number of ants in the colony (i hate ants!!!!)
EVAPORATION = 0.3               # how much pheromone evaporates each iteration
ITERATIONS = 10000              # number of iterations to run the algorithm
FEATURES_PER_ITERATION = 200    # number of features to select per iteration
BETA = 1                        # weight given for similarity vs pheromone
EXPLOITATION_RATE = 0.3         # how much to exploit vs explore
EPSILON = 0.00001               # Stop division by zero errors
SIMILARITY_FUNCTION = 'phi'     # similarity function to use, 'phi' or 'mi'

In [5]:
from urielplus import urielplus
from sklearn.metrics import matthews_corrcoef, normalized_mutual_info_score
import numpy as np
import time
import pandas as pd

# Initialization

In [6]:
uriel = urielplus.URIELPlus()
uriel.integrate_databases()
uriel.set_aggregation('U')
uriel.aggregate()

2025-05-29 15:30:33,233 - root - INFO - Importing all databases....
2025-05-29 15:30:33,234 - root - INFO - Importing updated SAPHON from "saphon_data.csv"....
2025-05-29 15:30:35,672 - root - INFO - Updated SAPHON integration complete..
2025-05-29 15:30:35,686 - root - INFO - Importing BDPROTO from "bdproto_data.csv"....
2025-05-29 15:30:35,686 - root - INFO - Converting ISO 639-3 codes to Glottocodes....
2025-05-29 15:30:35,837 - root - INFO - Conversion to Glottocodes complete.
2025-05-29 15:30:57,542 - root - INFO - BDPROTO integration complete.
2025-05-29 15:30:57,543 - root - INFO - Importing Grambank from "grambank_data.csv"....
2025-05-29 15:31:37,008 - root - INFO - Grambank integration complete.
2025-05-29 15:31:37,010 - root - INFO - Importing APiCS from "apics_data.csv"....
2025-05-29 15:31:39,995 - root - INFO - APiCS integration complete.
2025-05-29 15:31:39,997 - root - INFO - Importing eWAVE from "english_dialect_data.csv"....
2025-05-29 15:31:52,307 - root - INFO - eWA

array([[[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       [[-1.],
        [-1.],
        [-1.],
        ...,
        [-1.],
        [-1.],
        [-1.]],

       ...,

       [[ 1.],
        [ 0.],
        [ 0.],
        ...,
        [ 1.],
        [ 1.],
        [ 1.]],

       [[ 1.],
        [ 0.],
        [ 0.],
        ...,
        [ 1.],
        [ 1.],
        [ 1.]],

       [[ 1.],
        [ 0.],
        [ 0.],
        ...,
        [ 0.],
        [ 1.],
        [ 1.]]])

In [7]:
# Collect aggregated and imputed data
data: np.ndarray = np.squeeze(uriel.get_typological_data_array())
data.shape

(8174, 800)

# Similarity Functions

In [5]:
def phi_coefficient(x: np.ndarray, y: np.ndarray) -> float:
    """
    Calculate the absolute value of Matthews Correlation Coefficient (phi coefficient) for two binary vectors.

    Input
    -----
    - x: array-like (num_features,)
    - y: array-like (num_features,)

    Output
    ------
    The absolute value of the phi coefficient
    """
    return abs(matthews_corrcoef(x, y))

In [ ]:
def mutual_information(x: np.ndarray, y: np.ndarray) -> float:
    """
    Calculate the normalized mutual information between two binary variables.

    Input
    -----
    - x : array-like, shape (n,)
    - y : array-like, shape (n,)
    """
    return normalized_mutual_info_score(x, y)

# Helper Functions

In [7]:
def construct_weight_matrix(data: np.ndarray, similarity_function) -> np.ndarray:
    """
    Construct a weight (similarity) matrix from the data.

    Input
    -----
    - data: array-like (num_samples, num_features)
    - similarity_function: function to compute similarity between two features, follows the signature; func([ndarray], [ndarray]) -> float
    
    Output
    ------
    Square weight matrix of shape (num_features, num_features) where Wij is the similarity between feature i and feature j. 
    Wii is set to the maximum similarity value.
    """
    num_features = data.shape[1]
    weights = np.ones((num_features, num_features)) # Might not be 1 for MI and average aggregation
    
    for i in range(num_features):
        for j in range(i + 1, num_features):
            weights[i, j] = similarity_function(data[:, i], data[:, j])
            weights[j, i] = weights[i, j]
    
    return weights

construct_weight_matrix(data, phi_coefficient).shape

C:\Users\Chi\AppData\Local\Temp\ipykernel_11096\1891932296.py:43: RuntimeWarning: invalid value encountered in scalar divide
  nmi = (2 * mi) / (h_x + h_y)


(800, 800)

In [8]:
def construct_pheromone_matrix(data: np.ndarray) -> np.ndarray:
    """
    Construct a pheromone matrix from the data.

    Input
    -----
    - data: array-like, shape (n_samples, n_features)

    Output
    ------
    - pheromones: array-like, shape (n_features,)
    """
    num_features = data.shape[1]
    pheromones = np.ones((num_features,))
    
    return pheromones

construct_pheromone_matrix(data).shape

(800,)

# Unsupervised Feature Selection based on Ant Colony Optimization (UFSACO)
An unsupervised feature selection algorithm based on ant colony optimization (Tabakhi, 2014).

In [9]:
class Ant:
    """
    Class representing an ant in the Ant Colony Optimization algorithm.
    Each ant has a current feature and a set of selected features.
    """
    def __init__(self):
        self.current_feature: int = None
        self.selected_features: set[int] = set()

In [10]:
def choose_feature(ant: Ant, pheromones: np.ndarray, weights: np.ndarray, beta: float = 1.0, mode: str = 'prob') -> int:
    """
    Choose a feature for the ant to select based on pheromone levels and weights.
    
    Inputs
    ------
    - ant: Ant object representing the current ant
    - pheromones: array-like, shape (n_features,)
    - weights: array-like, shape (n_features, n_features)
    - beta: float, parameter controlling the influence of pheromone levels - default 1.0
    - mode: str, either 'prob' for probabilistic selection or 'greedy' for deterministic selection - default 'prob'

    Output
    ------
    The index of the selected feature.
    """
    num_features = pheromones.shape[0]

    if len(ant.selected_features) == num_features:
        raise ValueError("All features have been selected. Ensure that FEATURES_PER_ITERATION is less than the number of features.")

    mask = np.zeros(num_features, dtype=int)
    mask[list(ant.selected_features)] = 1

    logits = pheromones * ((1/(weights[ant.current_feature, :] + EPSILON)) ** beta)
    logits = np.where(mask == 0, logits, 0)

    if mode == 'prob': 
        probabilities: np.ndarray = logits / np.sum(logits)
        feature: int = np.random.choice(range(num_features), p=probabilities)
    elif mode == 'greedy':        
        feature: int = np.argmax(logits)
    else:
        raise ValueError("Invalid mode. Choose 'prob' or 'greedy'.")

    return feature

In [ ]:
def select_features_ACO(data: np.ndarray, similarity_function) -> np.ndarray:
    """
    Select features using the Ant Colony Optimization algorithm.
    The algorithm iteratively updates pheromone levels based on the features selected by the ants.

    Inputs
    ------
    - data: array-like, shape (n_samples, n_features)

    Outputs
    -------
    ndarray of features, ordered by final pheromone levels in descending order.
    """
    num_features = data.shape[1]

    weights: np.ndarray = construct_weight_matrix(data, similarity_function)
    pheromones: np.ndarray = construct_pheromone_matrix(data)
    ants: list[Ant] = [Ant() for _ in range(NUM_ANTS)]

    tik = time.time()
    for iteration in range(ITERATIONS):
        feature_count: np.ndarray = np.zeros((num_features), dtype=int)     # Holds number of times a feature has been selected this iteration

        # Initial placement of ants (no duplicates), does not count towards feature_count or pheromones
        placed_features = set()
        for ant in ants:
            feature: int = np.random.choice(list(set(range(num_features)) - placed_features))
            ant.current_feature = feature
            placed_features.add(feature)

        # Iterate
        for feature_choice in range(FEATURES_PER_ITERATION):
            for ant in ants:
                mode: str = 'prob' if np.random.rand() > EXPLOITATION_RATE else 'greedy'
                feature: int = choose_feature(ant, pheromones, weights, BETA, mode)
                ant.current_feature = feature
                ant.selected_features.add(feature)
                feature_count[feature] += 1
        
        # Update pheromones
        pheromones *= (1 - EVAPORATION)
        pheromones += (feature_count / np.sum(feature_count))

        # Reset ants for the next iteration
        for ant in ants:
            ant.current_feature = None
            ant.selected_features.clear()
        
        tok = time.time()
        if iteration % 100 == 0:
            print(f"Iteration {iteration}/{ITERATIONS}, Time elapsed: {tok - tik:.2f}s")
            tik = tok

    # Sort features by pheromone levels in descending order
    return np.argsort(pheromones)[::-1]

# Tests

In [12]:
""" Test select_features_ACO() """
# # Small binary feature dataset (6 samples, 5 features)
# test_data = np.array([
#     [1, 0, 1, 0, 1],  # Sample 1
#     [0, 1, 0, 1, 0],  # Sample 2
#     [1, 1, 1, 1, 0],  # Sample 3
#     [1, 1, 1, 1, 1],  # Sample 4
#     [1, 0, 1, 1, 1],  # Sample 5
#     [0, 0, 0, 0, 1],  # Sample 6
# ])
# test_data = test_data.T

# test_weights = construct_weight_matrix(test_data, phi_coefficient)
# test_pheromones = construct_pheromone_matrix(test_data)
# ants = [Ant() for _ in range(5)]
# print(test_weights)
# selected_features = select_features_ACO(ants, test_pheromones, test_weights)
# for i in selected_features:
#     print(f"Feature {i+1}: {test_data.T[i]}")

' Test select_features_ACO() '

In [13]:
""" Test choose_feature() """
# a = Ant()
# a.current_feature = 0
# a.selected_features = {1}

# test_weights = np.array([[1, 0.2, 0.3],
#                         [0.2, 1, 0.1],
#                         [0.3, 0.1, 1]])

# test_pheromones = np.array([0.5, 0.3, 0.2])
# np.random.seed(1)
# assert choose_feature(a, test_pheromones, test_weights) == 0

' Test choose_feature() '

# How to run

In [ ]:
similarity_function = phi_coefficient if SIMILARITY_FUNCTION == 'phi' else mutual_information
ranked_features = select_features_ACO(data, similarity_function)

Iteration 0, Time elapsed: 1.99 seconds
Iteration 100, Time elapsed: 167.23 seconds
Iteration 200, Time elapsed: 163.41 seconds
Iteration 300, Time elapsed: 161.73 seconds
Iteration 400, Time elapsed: 164.09 seconds
Iteration 500, Time elapsed: 163.29 seconds
Iteration 600, Time elapsed: 161.33 seconds
Iteration 700, Time elapsed: 179.33 seconds


KeyboardInterrupt: 

In [ ]:
data: np.ndarray = np.squeeze(uriel.get_typological_data_array())   # No imputation!
feature_labels = uriel.get_typological_features_array()
languages: np.ndarray = uriel.get_typological_languages_array()

df = pd.DataFrame(data, columns=feature_labels, index=languages)

In [19]:
for n_features in range(100, 701, 100):
    filtered_data: pd.DataFrame = df.iloc[:, ranked_features[:n_features]]
    filtered_data.to_csv(f'../selection_result/ant_{SIMILARITY_FUNCTION}_{n_features}.csv')